<a href="https://colab.research.google.com/github/keerthh129/AIML-Colab/blob/main/Random_forest_high.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
df = pd.read_csv('Heart.csv')

print(df.head())
print(df.columns)

   HeartRate     HRV  Age  Gender Activity  BodyTemperature StressLevel  \
0         98   79.89   36    Male  Walking            37.78      Medium   
1        111   89.48   45  Female  Walking            36.50        High   
2         88  108.05   43  Female  Walking            37.84         Low   
3         74   82.44   54    Male  Walking            37.32      Medium   
4        102   49.56   43    Male  Walking            37.77         Low   

   Emotion  
0     fear  
1    angry  
2    happy  
3  neutral  
4     fear  
Index(['HeartRate', 'HRV', 'Age', 'Gender', 'Activity', 'BodyTemperature',
       'StressLevel', 'Emotion'],
      dtype='object')


In [4]:
df = df.dropna()

In [5]:
le = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

In [6]:
target_column = 'Emotion'   # or 'emotion' or whatever you got

X = df.drop(target_column, axis=1)
y = df[target_column]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [8]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'bootstrap': [True, False]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

print("Best Parameters:", grid.best_params_)

Best Parameters: {'bootstrap': True, 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}


In [9]:
y_pred = best_model.predict(X_test)

In [10]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.975

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         3
           1       1.00      1.00      1.00        12
           2       1.00      0.83      0.91         6
           3       0.92      1.00      0.96        12
           4       1.00      1.00      1.00         7

    accuracy                           0.97        40
   macro avg       0.98      0.97      0.97        40
weighted avg       0.98      0.97      0.97        40


Confusion Matrix:
 [[ 3  0  0  0  0]
 [ 0 12  0  0  0]
 [ 0  0  5  1  0]
 [ 0  0  0 12  0]
 [ 0  0  0  0  7]]


In [11]:
cv_scores = cross_val_score(best_model, X, y, cv=5)
print("Cross-validation Accuracy:", np.mean(cv_scores))

Cross-validation Accuracy: 0.9400000000000001
